# Буткемп, день 4 — отказоустойчивый сборщик стакана

Задание дня — сделать сборщик стакана failure-mode-aware на двух днях perpetual L2 трёх бирж (Bybit, OKX, Binance). Слайд требует дословно пять вещей:

1. Detect & recover gaps — найти разрывы и сбросы последовательности по каждой бирже, пересобраться от снапшота и никогда не выдать стакан через невосстановленный разрыв.
2. Flag impossible books — поймать crossed, locked и устаревшие стаканы по бирже и по часу, объяснить паттерн.
3. Measure feed health — задержку отправка→приём и её выбросы, кросс-биржевой сдвиг часов и случай, когда одни таймстемпы переупорядочивают события.
4. Post-trade on simulated fills — пройти стакан для ордеров нескольких размеров при заявленной задержке, slippage относительно decision mid и arrival mid, cost vs size, markouts на 100 мс, 1 с, 10 с.
5. Inject & survive — испортить копию фида (drop, duplicate, reorder, reset) и показать, что набор инвариантов ловит каждый случай.

По футеру слайда: у нас только L2, без tape и без собственных сделок, поэтому busts, STP, пассивные markouts и реальные комиссии в оценку не входят. Симулированные исполнения не учитывают собственное влияние на рынок и любую задержку, кроме заявленной, так что все издержки ниже — верхняя граница.

Данные — три файла за 1–3 июля 2026 (перпетуалы, BTC):

| файл | биржа | символ | строк |
|---|---|---|---|
| `bybit_l2_20260701_20260703.parquet` | Bybit | `BTCUSDT` | 63.2 млн |
| `binance_l2_20260701_20260703.parquet` | Binance | `BTCUSDT` | 369.3 млн |
| `okex_l2_20260701_20260703.parquet` | OKX | `BTC-USDT-SWAP` | 209.2 млн |

Период у всех — с 03:00 1 июля до 03:00 3 июля, но первый снапшот появляется только около 09:28–09:31, поэтому до него строить нечего: реконструкция начинается от снапшота. Схема одинаковая у трёх бирж: `entry_type=1` — снапшот, `entry_type=2` — инкремент, `entry_type=3` — heartbeat; `_feed` 1/2/3 — подфиды одного фида; `side` 1 — bid, −1 — ask; `amount<=0` — снятие уровня. Отдельно выдан независимый эталон `l2_executable_quotes_100ms_10000.parquet` — top-of-book по всем трём биржам на 100-мс сетке; по нему проверяем сборщик.

## Разведка данных

Задача позволяет смотреть на отрезках, поэтому все тяжёлые построения книги считаются на окне, а по полному периоду идут только агрегаты (счётчики, латентность, разрывы) — они выполняются за секунды-минуты. Окно задаётся одной парой переменных, и его можно растянуть на весь период, ничего не меняя в коде.

Смотрим в первую очередь на то, что определяет отказоустойчивость: сколько строк какого типа, сколько подфидов, где снимается `sequence_no`, когда случаются снапшоты и что показывают колонки времени.

In [1]:
import duckdb, numpy as np, pandas as pd, time, gc
from sortedcontainers import SortedDict

# Сборщик копит длинные списки снимков и состояний; циклов в них нет,
# поэтому автоматический GC только тратит время на обходы миллионов объектов.
gc.disable()

DATA = '/Users/b.kotov/Downloads'
SRC = {
    'bybit':   (f'{DATA}/bybit_l2_20260701_20260703.parquet',   'BTCUSDT'),
    'binance': (f'{DATA}/binance_l2_20260701_20260703.parquet', 'BTCUSDT'),
    'okx':     (f'{DATA}/okex_l2_20260701_20260703.parquet',    'BTC-USDT-SWAP'),
}
REF = f'{DATA}/l2_executable_quotes_100ms_10000.parquet'

WINDOW_START  = '2026-07-01 09:32:00+03'   # сразу после первого снапшота каждой биржи
WINDOW_HOURS  = 2.5                        # окно построения книги, часы (48 = весь период)
GRID_MS       = 50                         # каденс снимков глубины для симуляции филлов
PAUSE_S       = 1.0                        # пауза приёма, ниже которой скачок номера — не потеря
STALE_TAU     = 0.25                       # порог устаревания, с (пауза между пачками)
PRICE_LO, PRICE_HI = 1000.0, 500_000.0     # окно валидных цен BTC

W0 = pd.Timestamp(WINDOW_START)
W1 = W0 + pd.Timedelta(hours=WINDOW_HOURS)

con = duckdb.connect()
con.execute("SET threads=6")
con.execute("SET memory_limit='8GB'")
con.execute("SET temp_directory='/tmp/duckdb_tmp'")
con.execute("SET max_temp_directory_size='6GB'")
print('duckdb', duckdb.__version__, '| окно', W0, '->', W1)

duckdb 1.5.6 | окно 2026-07-01 09:32:00+03:00 -> 2026-07-01 12:02:00+03:00


In [2]:
print(con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [SRC['bybit'][0]]).df()
      [['column_name','column_type']].to_string(index=False))

          column_name              column_type
               symbol                  VARCHAR
           event_time TIMESTAMP WITH TIME ZONE
     event_time_nanos                  INTEGER
         insertion_no                   BIGINT
        exchange_time TIMESTAMP WITH TIME ZONE
  exchange_time_nanos                  INTEGER
          sequence_no                   BIGINT
                 side                  INTEGER
                price                   DOUBLE
               amount                   DOUBLE
           entry_type                  INTEGER
        entry_markers                  INTEGER
      _insertion_time TIMESTAMP WITH TIME ZONE
_insertion_time_nanos                  INTEGER
             _gate_id                  INTEGER
                _feed                  INTEGER
       _feed_l2_depth                  INTEGER
             exchange                  VARCHAR


In [3]:
for name, (path, sym) in SRC.items():
    print('='*60, name, sym)
    print(con.execute("""
        SELECT entry_type, _feed, count(*) n, count(sequence_no) n_seq
        FROM read_parquet(?) WHERE symbol=? AND event_time>? AND event_time<=?
        GROUP BY 1,2 ORDER BY 1,2""", [path, sym, W0, W1]).df().to_string(index=False))

============================================================ bybit BTCUSDT
 entry_type  _feed       n   n_seq
          2      1 2536419 2536419
          2      2  219623  219623
          2      3   32750       0
============================================================ binance BTCUSDT


 entry_type  _feed        n    n_seq
          2      1 13171530 13171530
          2      2  3463796  3463796
          2      3    86136        0
          2      4      585      585
============================================================ okx BTC-USDT-SWAP


 entry_type  _feed        n    n_seq
          2      1 10074425 10074425
          2      2   110600   110600
          2      3    80687        0


In [4]:
for name, (path, sym) in SRC.items():
    snap = con.execute("""
        SELECT _feed, event_time, count(*) n, count(DISTINCT sequence_no) n_seq
        FROM read_parquet(?) WHERE symbol=? AND entry_type=1
        GROUP BY 1,2 ORDER BY 2""", [path, sym]).df()
    print(name, '| снапшотов:', len(snap), '| времена:', list(snap.event_time.astype(str))[:6])

bybit | снапшотов: 6 | времена: ['2026-07-01 09:31:05.480956+03:00', '2026-07-01 21:10:24.904450+03:00', '2026-07-01 21:10:24.908689+03:00', '2026-07-01 22:02:45.115923+03:00', '2026-07-01 22:50:12.300959+03:00', '2026-07-01 22:58:32.519056+03:00']


binance | снапшотов: 1 | времена: ['2026-07-01 09:29:21.090047+03:00']
okx | снапшотов: 2 | времена: ['2026-07-01 09:28:26.729801+03:00', '2026-07-01 09:28:26.753905+03:00']


In [5]:
# _insertion_time — это штамп батча доставки, а не время строки: по нему «задержка» выходит отрицательной.
print(con.execute("""
    SELECT round(quantile_cont(epoch(_insertion_time-exchange_time),0.5),4) p50_insertion,
           round(quantile_cont(epoch(event_time-exchange_time),0.5),4)        p50_event,
           count(DISTINCT _insertion_time) n_stamp, count(*) n_rows
    FROM read_parquet(?) WHERE symbol=? AND entry_type=2 AND _feed=1 AND event_time>? AND event_time<=?""",
    [SRC['bybit'][0], 'BTCUSDT', W0, W0+pd.Timedelta(minutes=1)]).df().to_string(index=False))

 p50_insertion  p50_event  n_stamp  n_rows
       -2.1394     0.0373       11   14596


### Порог разрыва по данным

Разрыв — это пропуск сообщений в нумерации подфида. Обычный шаг `sequence_no` между соседними сообщениями не равен единице: биржа пакует в один номер целую дельту, а сам номер разрежен произвольно, поэтому шаг болтается от единиц до десятков тысяч. Порог «шаг больше X — потеря» нельзя назначить константой: у Bybit p99.99 шага подфида 2 — 41 тыс., у OKX — 6 тыс.; одна граница на всех даст либо ложные тревоги на Bybit, либо пропуск реальных разрывов на OKX.

Второй признак — пауза приёма. Настоящая потеря дельт отзывается паузой в потоке (пока фид молчал, биржа успела уйти вперёд), а большой шаг при нормальном 20-мс каденсе — это просто толстая дельта, а не потеря. Поэтому порог берём по данным: максимальный шаг, наблюдённый при паузе приёма не больше секунды, плюс единица; и сама пауза при этом должна быть больше секунды. Такой порог не даёт ложных срабатываний по построению — всё, что при короткой паузе его не превысило, физически неотличимо от плотной дельты.

Считаем распределение шага и калибруем порог по полному периоду, отдельно каждому подфиду.

In [6]:
GAP_THR = {}
cal_rows = []
for name, (path, sym) in SRC.items():
    df = con.execute("""
    WITH u AS (
      SELECT _feed f, sequence_no s, min(file_row_number) frn, min(event_time) t
      FROM read_parquet(?, file_row_number=true)
      WHERE symbol=? AND entry_type=2 AND _feed IN (1,2) AND sequence_no IS NOT NULL
      GROUP BY 1,2),
    m AS (
      SELECT f, s - lag(s) OVER (PARTITION BY f ORDER BY frn) d,
             epoch(t - lag(t) OVER (PARTITION BY f ORDER BY frn)) dt
      FROM u)
    SELECT f, count(*) n_msg,
           round(quantile_cont(d, 0.5),0)  p50,
           round(quantile_cont(d, 0.99),0) p99,
           round(quantile_cont(d, 0.9999) FILTER (WHERE d>0 AND dt<=?),0) p9999,
           max(d) FILTER (WHERE d>0 AND dt<=?) max_normal
    FROM m GROUP BY 1 ORDER BY 1""", [path, sym, PAUSE_S, PAUSE_S]).df()
    GAP_THR[name] = {}
    for r in df.itertuples(index=False):
        GAP_THR[name][int(r.f)] = int(r.max_normal) + 1
        cal_rows.append(dict(venue=name, feed=int(r.f), n_msg=int(r.n_msg), p50=int(r.p50),
                             p99=int(r.p99), p9999=int(r.p9999),
                             max_normal=int(r.max_normal), threshold=GAP_THR[name][int(r.f)]))
calib = pd.DataFrame(cal_rows)
print(calib.to_string(index=False))

  venue  feed    n_msg  p50   p99  p9999  max_normal  threshold
  bybit     1  6189202  692  4297  16152       43834      43835
  bybit     2  3013947 1044 15488  40919       59711      59712
binance     1  6421057 2259 14473  26297       55416      55417
binance     2 76879091   22  4669  26661      113833     113834
    okx     1 13221421   10   612   3454       10385      10386
    okx     2  3129893   58  1656   6374       63012      63013


## Сборщик и набор инвариантов

Сборщик держит две отсортированные лестницы уровней — bid и ask — и применяет к ним снапшот как базу, потом инкременты в порядке `(event_time, file_row_number)`. Ключ в bid-лестнице отрицательный (`-p`), чтобы лучший bid был первым элементом; `amount<=0` снимает уровень. Порядок внутри одного `event_time` берётся по физической позиции строки в файле, потому что `sequence_no` у разных подфидов — разные пространства и сортировка по нему ломает книгу.

Поток приходит пачками: строки с одинаковым `event_time` — это один батч доставки, внутри которого биржа успела провести несколько сообщений. Публикуем состояние ровно на конце пачки: в момент, когда следующий батч уже пришёл с новым `event_time`. Внутри пачки книга проходит через промежуточные состояния, и часть из них crossed: разные подфиды приносят установку и снятие встречных уровней в разные строки одной пачки. Это артефакт порядка применения внутри батча, а не наблюдаемое рынком состояние: наружу такое состояние не отдаётся.

Проверка последовательности идёт на уровне сообщения: у инкрементов один `sequence_no` стоит на всех строках дельты, поэтому новым сообщением считается только смена номера. Шаг вверх больше калиброванного порога вместе с паузой приёма больше секунды означает разрыв (данные потеряны, книга устарела — торговать нельзя), номер назад — либо дубликат уже виденного сообщения, либо сброс (биржа переоткрыла поток, нужен снапшот). Флагов два, потому что последствия разрыва в разных подфидах неодинаковы. Разрыв или сброс базового `feed1` блокирует выдачу: `feed1` несёт базу стакана и основной поток изменений, и до следующего снапшота отличить потерянные обновления от разреженного счётчика невозможно — состояние помечается непригодным (`healthy=False`) и не идёт в расчёты. Тот же случай во вторичном `feed2` базу не затрагивает: он искажает глубину, но не best bid/ask, поэтому выдачу не блокирует, а только помечает состояние меткой `degraded`. Оба флага снимает ближайший снапшот.

Латентность считается с наносекундной точностью: $\ell = \big(\mathrm{epoch}(T^{recv}) + n^{recv}/10^9\big) - \big(\mathrm{epoch}(T^{exch}) + n^{exch}/10^9\big)$, где $T^{recv}$ и $n^{recv}$ — наши `event_time` и `event_time_nanos`, а $T^{exch}$ и $n^{exch}$ — биржевые `exchange_time` и `exchange_time_nanos`. `event_time` усечён до миллисекунд, а микросекунды живут только в отдельной колонке `event_time_nanos`, так что без неё приёмные штампы округлялись бы до миллисекунды и размывали медиану задержки Binance, равную 2.3 мс. Биржевой `exchange_time_nanos` при этом тождественно нулевой у всех трёх бирж, то есть биржевые штампы в дампе миллисекундные и наносекунды уточняют только нашу сторону; в пункте 3 это подтверждается числами. Slippage ордера с benchmark $M$ — это $s = \mathrm{side}\cdot(P_{fill}-M)/M$; markout через $\Delta$ после исполнения — $\mathrm{mo}(\Delta)=q\,(M(t_{fill}+\Delta)-P_{fill})/M_{decision}$, где $q=+1$ для покупки и $-1$ для продажи.

Набор инвариантов: хронология таймстемпов, отсутствие разрывов и сбросов последовательности, отсутствие дубликатов сообщений, отсутствие crossed-книги на концах пачек и сходимость позиций (снятие уровня, которого нет в книге, — редкость). Для пункта 5 инварианты проверяются как отклонение от чистого прогона, а не как абсолют: в реальном окне есть разрывы, и набор, проваливающийся на чистом фиде, ничего не измеряет.

In [7]:
def iter_batches(path, sym, t_lo, t_hi, feeds=(1, 2, 3)):
    # Батчи строк инкрементов в порядке (event_time, file_row_number) — потоково, крупными порциями.
    feed_in = ','.join(str(f) for f in feeds)
    res = con.execute(
        "SELECT event_time t, _feed feed, sequence_no seq, side, price, amount "
        "FROM read_parquet(?, file_row_number=true) "
        f"WHERE symbol=? AND entry_type=2 AND _feed IN ({feed_in}) AND event_time>? AND event_time<=? "
        "ORDER BY event_time, file_row_number", [path, sym, t_lo, t_hi])
    while True:
        chunk = res.fetch_df_chunk(100)
        if chunk.shape[0] == 0:
            return
        yield (chunk['t'].tolist(), chunk['feed'].tolist(), chunk['seq'].tolist(),
               chunk['side'].tolist(), chunk['price'].tolist(), chunk['amount'].tolist())


def seed_book(path, sym, t0):
    # Стакан из последнего снапшота не позже t0.
    snap_t = con.execute("""
        SELECT max(event_time) FROM read_parquet(?)
        WHERE symbol=? AND _feed=1 AND entry_type=1 AND event_time<=?""", [path, sym, t0]).fetchone()[0]
    if snap_t is None:
        snap_t = con.execute("""
            SELECT min(event_time) FROM read_parquet(?)
            WHERE symbol=? AND _feed=1 AND entry_type=1""", [path, sym]).fetchone()[0]
    snap = con.execute("""
        SELECT sequence_no, side, price, amount FROM read_parquet(?)
        WHERE symbol=? AND _feed=1 AND entry_type=1 AND event_time=?""", [path, sym, snap_t]).df()
    bids, asks = SortedDict(), SortedDict()
    for s, p, a in snap[['side','price','amount']].itertuples(index=False):
        if a and a > 0:
            (bids if s == 1 else asks)[-p if s == 1 else p] = a
    return bids, asks, int(snap['sequence_no'].iloc[0]), snap_t


def load_snapshots(path, sym, t_after, t_until):
    # Снапшоты строго после сида — точки пересборки книги.
    df = con.execute("""
        SELECT event_time, sequence_no, side, price, amount FROM read_parquet(?)
        WHERE symbol=? AND _feed=1 AND entry_type=1 AND event_time>? AND event_time<=?
        ORDER BY event_time""", [path, sym, t_after, t_until]).df()
    out = []
    for st, g in df.groupby('event_time'):
        bd, ad = SortedDict(), SortedDict()
        for side, price, amount in g[['side','price','amount']].itertuples(index=False):
            if amount and amount > 0:
                (bd if side == 1 else ad)[-price if side == 1 else price] = amount
        out.append((st, bd, ad, int(g['sequence_no'].iloc[0])))
    return out


def reconstruct(batches, bids, asks, snap_seq, seq_feeds, t_record_from, t_end,
                gap_thr, pause_s=PAUSE_S, resync=None, depth_levels=0, depth_from=None):
    last_seq = {f: snap_seq for f in seq_feeds}
    last_t   = {f: None for f in seq_feeds}
    seen     = {f: set() for f in seq_feeds}
    resync = resync or []
    ri = 0
    stats = dict(msg=0, dup_rows=0, dup_msg=0, gap=0, reset=0, orphan=0, invalid=0,
                 unhealthy_out=0, degraded_out=0, resyncs=0, rows=0, crossed_rows=0,
                 packets=0, packets_crossed=0,
                 max_bid_levels=0, max_ask_levels=0, sum_bid_levels=0, sum_ask_levels=0)
    healthy = True      # False, пока не закрыт разрыв или сброс базового feed1
    degraded = False    # True после гэпа вторичного подфида; выдачу не блокирует
    gap_events, dup_events = [], []
    tops, depths = [], []
    prev_t = None
    for t_l, f_l, q_l, s_l, p_l, a_l in batches:
        for i in range(len(t_l)):
            t = t_l[i]; f = f_l[i]; seq = q_l[i]; s = s_l[i]; p = p_l[i]; a = a_l[i]
            while ri < len(resync) and t >= resync[ri][0]:
                sb, sa, sq = resync[ri][1], resync[ri][2], resync[ri][3]
                bids.clear(); bids.update(sb); asks.clear(); asks.update(sa)
                for ff in last_seq: last_seq[ff] = sq
                healthy = True; degraded = False; ri += 1; stats['resyncs'] += 1
            if not (PRICE_LO < p < PRICE_HI) or a is None:
                stats['invalid'] += 1
                continue
            if f in last_seq and seq is not None:
                d = seq - last_seq[f]
                if d == 0:
                    stats['dup_rows'] += 1        # та же дельта, ещё одна строка
                else:
                    stats['msg'] += 1
                    pause = (t - last_t[f]).total_seconds() if last_t[f] is not None else None
                    if d < 0:
                        if seq in seen[f]:
                            stats['dup_msg'] += 1; dup_events.append((t, f, seq))
                        else:
                            stats['reset'] += 1
                        # разрыв базового подфида блокирует выдачу, вторичного — только помечает
                        if f == 1: healthy = False
                        else: degraded = True
                    elif d > gap_thr.get(f, 10**9) and (pause is None or pause > pause_s):
                        stats['gap'] += 1
                        gap_events.append((t, f, seq, d, pause))
                        if f == 1: healthy = False
                        else: degraded = True
                    last_seq[f] = seq; seen[f].add(seq); last_t[f] = t
            k = -p if s == 1 else p
            book = bids if s == 1 else asks
            if a <= 0:
                if k not in book: stats['orphan'] += 1
                book.pop(k, None)
            else:
                book[k] = a
            stats['rows'] += 1
            if bids and asks and -bids.peekitem(0)[0] > asks.peekitem(0)[0]:
                stats['crossed_rows'] += 1        # crossed внутри пачки — промежуточное состояние
            if prev_t is not None and t != prev_t:
                # конец пачки — единственный момент, когда состояние публикуется
                if bids and asks:
                    stats['packets'] += 1
                    bb, ba = bids.peekitem(0), asks.peekitem(0)
                    cr = -bb[0] > ba[0]
                    if cr: stats['packets_crossed'] += 1
                    lb, la = len(bids), len(asks)
                    stats['max_bid_levels'] = max(stats['max_bid_levels'], lb)
                    stats['max_ask_levels'] = max(stats['max_ask_levels'], la)
                    stats['sum_bid_levels'] += lb; stats['sum_ask_levels'] += la
                    if t_record_from <= prev_t <= t_end:
                        if not healthy: stats['unhealthy_out'] += 1
                        if degraded: stats['degraded_out'] += 1
                        tops.append((prev_t, -bb[0], bb[1], ba[0], ba[1], healthy, degraded, cr))
                        if depth_levels and depth_from is not None and prev_t >= depth_from:
                            depths.append((prev_t,
                                           [(-q2, w) for q2, w in bids.items()[:depth_levels]],
                                           [(q2, w) for q2, w in asks.items()[:depth_levels]]))
            prev_t = t
    return dict(tops=tops, depths=depths, stats=stats,
                gap_events=gap_events, dup_events=dup_events, last_seq=last_seq)


def build_book(path, sym, t_record_from, t_end, resync=True, **kw):
    bids, asks, snap_seq, snap_t = seed_book(path, sym, t_record_from)
    snaps = load_snapshots(path, sym, snap_t, t_end) if resync else []
    out = reconstruct(iter_batches(path, sym, snap_t, t_end), bids, asks, snap_seq,
                      seq_feeds=(1, 2), t_record_from=t_record_from, t_end=t_end,
                      resync=snaps, **kw)
    out['seed_time'] = snap_t
    return out


def tops_frame(out):
    return pd.DataFrame(out['tops'],
                        columns=['t','bid','bid_sz','ask','ask_sz','healthy','degraded','crossed'])

In [8]:
BOOKS = {}
depth_from = W1 - pd.Timedelta(hours=1)

for name, (path, sym) in SRC.items():
    tic = time.time()
    out = build_book(path, sym, W0, W1, depth_levels=100, depth_from=depth_from,
                     gap_thr=GAP_THR[name])
    BOOKS[name] = out
    st = out['stats']
    print(f"{name:<8} packets={st['packets']:>8} crossed@pkt={st['packets_crossed']:>6}  "
          f"depth={len(out['depths']):>7}  msg={st['msg']:>8}  dup_rows={st['dup_rows']:>9}  "
          f"gap={st['gap']}  reset={st['reset']}  dup_msg={st['dup_msg']}  "
          f"orphan={st['orphan']:>6}  invalid={st['invalid']:>5}  "
          f"blocked={st['unhealthy_out']}  degraded={st['degraded_out']}  t={time.time()-tic:.0f}s")

bybit    packets=  483339 crossed@pkt=  3913  depth= 192175  msg=  474729  dup_rows=  2295668  gap=14  reset=0  dup_msg=0  orphan=     1  invalid=    0  blocked=209897  degraded=399785  t=21s


binance  packets= 1543357 crossed@pkt= 42445  depth= 619460  msg= 3779609  dup_rows= 13075473  gap=0  reset=0  dup_msg=0  orphan=     3  invalid= 8561  blocked=0  degraded=0  t=95s


okx      packets=  792877 crossed@pkt=   793  depth= 311411  msg=  782607  dup_rows=  9597644  gap=0  reset=0  dup_msg=0  orphan=     0  invalid=    0  blocked=0  degraded=0  t=58s


### Проверка сборщика против независимого эталона

Восстановленный top-of-book надо сверить с эталоном, а не с самим собой. Эталон даёт bid и ask на 100-мс сетке; для каждой его точки берём последнее состояние сборщика не позже этой точки (backward-соединение) и сравниваем обе котировки побитово. Совпадение ниже 100 % ожидаемо: сетка эталона выбрасывает часть тактов, и пересечение фронтов top-of-book попадает в соседнюю ячейку.

In [9]:
ref = con.execute("""
    SELECT event_time, exchange, mid, best_bid, best_ask FROM read_parquet(?)
    WHERE asset='BTC' AND event_time>? AND event_time<=? ORDER BY event_time""",
    [REF, W0, W1]).df()

PAIR = {'Bybit': 'bybit', 'Binance': 'binance', 'Okex': 'okx'}
match_rows = []
for ex, name in PAIR.items():
    state = tops_frame(BOOKS[name])[['t','bid','ask']].rename(columns={'t':'event_time'})
    r = ref[ref.exchange == ex]
    m = pd.merge_asof(r, state, on='event_time', direction='backward')
    ok = np.isclose(m.best_bid, m.bid, atol=1e-9) & np.isclose(m.best_ask, m.ask, atol=1e-9)
    match_rows.append(dict(venue=name, ref_points=len(r), match=float(ok.mean()),
                           bid=float(np.isclose(m.best_bid, m.bid, atol=1e-9).mean()),
                           ask=float(np.isclose(m.best_ask, m.ask, atol=1e-9).mean())))
match = pd.DataFrame(match_rows)
print(match.to_string(index=False, float_format=lambda v: f'{v:.4%}'))

  venue  ref_points    match      bid      ask
  bybit       87023 99.7633% 99.9713% 99.7920%
binance       89960 97.1621% 99.2230% 97.9391%
    okx       89549 99.8760% 99.9062% 99.9698%


### Пункт 1 — Detect & recover gaps

Разрыв ищем по номеру сообщения внутри подфида: шаг вверх больше калиброванного порога и пауза приёма больше секунды — потеряны дельты; номер ушёл назад — дубликат или сброс. Гэп базового `feed1` означает, что книга рассинхронизирована, и её нельзя показывать, пока не пришёл следующий снапшот; тот же случай во вторичном `feed2` книгу не рассинхронизирует, но помечает состояние. Прогоняем детекцию двумя способами: оконной функцией по калиброванному порогу считаем события, а по записанному потоку состояний проверяем, что через невосстановленный гэп не прошла ни одна выдача.


In [10]:
gap_scan = {}
for name, (path, sym) in SRC.items():
    tic = time.time()
    thr = GAP_THR[name]
    q = """
    WITH u AS (   -- одно сообщение = один sequence_no у подфида; строк в сообщении десятки
      SELECT _feed, sequence_no, min(file_row_number) frn, min(event_time) t
      FROM read_parquet(?, file_row_number=true)
      WHERE symbol=? AND entry_type=2 AND _feed IN (1,2) AND sequence_no IS NOT NULL
      GROUP BY 1,2),
    m AS (
      SELECT _feed, sequence_no,
             sequence_no - lag(sequence_no) OVER (PARTITION BY _feed ORDER BY frn) d,
             epoch(t - lag(t) OVER (PARTITION BY _feed ORDER BY frn)) dt
      FROM u)
    SELECT _feed, count(*) n_msg,
           sum(CASE WHEN d<0 THEN 1 ELSE 0 END) decreases,
           sum(CASE WHEN d > CASE _feed WHEN 1 THEN ? WHEN 2 THEN ? END AND dt > ? THEN 1 ELSE 0 END) gaps
    FROM m GROUP BY 1 ORDER BY 1"""
    df = con.execute(q, [path, sym, thr[1], thr[2], PAUSE_S]).df()
    gap_scan[name] = df
    print(name, f'({time.time()-tic:.0f}s)')
    print(df.to_string(index=False))

bybit (6s)
 _feed   n_msg  decreases  gaps
     1 6189202        0.0   8.0
     2 3013947        0.0 328.0


binance (61s)
 _feed    n_msg  decreases  gaps
     1  6421057        0.0   1.0
     2 76879091        0.0   1.0


okx (16s)
 _feed    n_msg  decreases  gaps
     1 13221421        0.0   1.0
     2  3129893        0.0   0.0


### Пункт 1 — независимая проверка «не выдавать стакан через разрыв»

Счётчик нездоровых выдач — это внутренняя бухгалтерия сборщика, и проверять ею же корректность нельзя. Построим интервалы «момент разрыва → ближайший снапшот» и проверим по записанному потоку состояний, что ни одно пригодное состояние не попало внутрь такого интервала. Блокирует выдачу только разрыв базового подфида, поэтому интервалы строим по событиям `feed1`, а гэпы `feed2` считаем отдельно: они выдачу не останавливают, а лишь помечают состояния меткой `degraded`. Снапшоты Bybit приходят редко — раз в час-полтора, — поэтому разрыв без следующего снапшота держит книгу замороженной до конца окна; это и есть плата за требование «никогда не выдать стакан через разрыв».

Заодно сверим номер каждого снапшота с последним номером инкрементов: сброс нумерации виден как шаг назад.


In [11]:
rows, counts = [], []
for name, (path, sym) in SRC.items():
    snap_ts = con.execute("""
        SELECT event_time FROM read_parquet(?)
        WHERE symbol=? AND _feed=1 AND entry_type=1 ORDER BY event_time""", [path, sym]).df()['event_time']
    st = pd.to_datetime(snap_ts).astype('int64').to_numpy()
    df = tops_frame(BOOKS[name])
    tv = pd.to_datetime(df['t']).astype('int64').to_numpy()
    ok = df['healthy'].to_numpy()
    n_block = n_mark = 0
    for (gt, f, seq, d, pause) in BOOKS[name]['gap_events']:
        if f != 1:                      # гэп вторичного подфида выдачу не блокирует
            n_mark += 1
            continue
        n_block += 1
        g = pd.Timestamp(gt).value
        j = int(np.searchsorted(st, g, side='right'))
        if j >= len(st):
            rows.append(dict(venue=name, gap=str(gt)[11:23], feed=f, delta=int(d),
                             pause=None if pause is None else round(pause,2),
                             until='—', usable=0))
            continue
        end = st[j]
        inside = (tv > g) & (tv < end)
        rows.append(dict(venue=name, gap=str(gt)[11:23], feed=f, delta=int(d),
                         pause=None if pause is None else round(pause,2),
                         until=str(snap_ts.iloc[j])[11:23],
                         usable=int((ok & inside).sum())))
    counts.append((name, n_block, n_mark))

print('гэпы, которые могут блокировать выдачу (feed1), и маркирующие (feed2):')
for name, nb_, nm_ in counts:
    print(f'  {name:<8} feed1: {nb_:>3}   feed2: {nm_:>3}')
print()
gap_check = pd.DataFrame(rows)
print('интервалы разрыв feed1 -> снапшот и пригодные состояния внутри:')
print(gap_check.to_string(index=False) if len(gap_check) else '  (блокирующих гэпов нет)')
print()
print('пригодных состояний внутри невосстановленных интервалов:',
      int(gap_check['usable'].sum()) if len(gap_check) else 0)


гэпы, которые могут блокировать выдачу (feed1), и маркирующие (feed2):
  bybit    feed1:   1   feed2:  13
  binance  feed1:   0   feed2:   0
  okx      feed1:   0   feed2:   0

интервалы разрыв feed1 -> снапшот и пригодные состояния внутри:
venue          gap  feed  delta  pause until  usable
bybit 10:55:57.543     1  46025    1.2     —       0

пригодных состояний внутри невосстановленных интервалов: 0


In [12]:
# Номер снапшота против последнего номера инкрементов: сброс нумерации = шаг назад.
for name, (path, sym) in SRC.items():
    sn = con.execute("""
        SELECT event_time t, sequence_no seq, count(*) n_lev FROM read_parquet(?)
        WHERE symbol=? AND _feed=1 AND entry_type=1 GROUP BY 1,2 ORDER BY 1""", [path, sym]).df()
    print('='*40, name, '| снапшотов:', len(sn))
    for r in sn.itertuples(index=False):
        lastinc = con.execute("""
            SELECT max(sequence_no) FROM read_parquet(?)
            WHERE symbol=? AND _feed=1 AND entry_type=2 AND event_time<=?""",
            [path, sym, r.t]).fetchone()[0]
        d = None if lastinc is None else int(r.seq) - int(lastinc)
        tag = '  <- сброс нумерации' if (d is not None and d < 0) else ''
        print(f"  {str(r.t)[:23]}  seq={int(r.seq)}  последний инкр.={lastinc}  "
              f"Δ={'—' if d is None else d}  уровней={int(r.n_lev)}{tag}")

======================================== bybit | снапшотов: 5


  2026-07-01 09:31:05.480  seq=646554928143  последний инкр.=646554936979  Δ=-8836  уровней=100  <- сброс нумерации


  2026-07-01 21:10:24.908  seq=647957933170  последний инкр.=647957945374  Δ=-12204  уровней=100  <- сброс нумерации


  2026-07-01 22:02:45.115  seq=648069606195  последний инкр.=648069609029  Δ=-2834  уровней=100  <- сброс нумерации


  2026-07-01 22:50:12.300  seq=648168673459  последний инкр.=648168674101  Δ=-642  уровней=100  <- сброс нумерации


  2026-07-01 22:58:32.519  seq=648186217080  последний инкр.=648186219723  Δ=-2643  уровней=100  <- сброс нумерации


======================================== binance | снапшотов: 1


  2026-07-01 09:29:21.090  seq=10937614462375  последний инкр.=10937614462754  Δ=-379  уровней=200  <- сброс нумерации
======================================== okx | снапшотов: 1


  2026-07-01 09:28:26.753  seq=327425058537  последний инкр.=327425058564  Δ=-27  уровней=800  <- сброс нумерации


In [13]:
# Примеры разрывов: время, величина скачка и пауза приёма.
gap_ex = con.execute("""
  WITH u AS (
    SELECT _feed, sequence_no, min(file_row_number) frn, min(event_time) t
    FROM read_parquet(?, file_row_number=true)
    WHERE symbol=? AND entry_type=2 AND _feed IN (1,2) AND sequence_no IS NOT NULL
    GROUP BY 1,2),
  m AS (
    SELECT _feed, t, sequence_no,
           sequence_no - lag(sequence_no) OVER (PARTITION BY _feed ORDER BY frn) d,
           epoch(t - lag(t) OVER (PARTITION BY _feed ORDER BY frn)) dt FROM u)
  SELECT _feed, t, d, round(dt,3) pause FROM m
  WHERE d > CASE _feed WHEN 1 THEN ? WHEN 2 THEN ? END AND dt > ?
  ORDER BY t LIMIT 20""",
  [SRC['bybit'][0], 'BTCUSDT', GAP_THR['bybit'][1], GAP_THR['bybit'][2], PAUSE_S]).df()
print('первые разрывы Bybit (порог + пауза):', len(gap_ex))
print(gap_ex.to_string(index=False))

первые разрывы Bybit (порог + пауза): 20
 _feed                                t      d  pause
     2 2026-07-01 03:02:44.775030+03:00  64251  1.180
     2 2026-07-01 03:07:27.714886+03:00  70332  1.250
     2 2026-07-01 03:10:56.495063+03:00  67110  1.280
     2 2026-07-01 03:15:36.784855+03:00  81734  1.810
     2 2026-07-01 03:21:03.114731+03:00  75542  1.679
     2 2026-07-01 03:37:42.394648+03:00  77394  1.700
     2 2026-07-01 03:38:09.774917+03:00  60142  1.560
     2 2026-07-01 03:42:56.855069+03:00 128768  3.580
     2 2026-07-01 03:43:00.234960+03:00 126370  3.380
     2 2026-07-01 03:43:10.274790+03:00  74842  1.820
     2 2026-07-01 03:43:26.398021+03:00  98271  1.883
     2 2026-07-01 03:47:52.764930+03:00  74394  1.290
     2 2026-07-01 03:48:14.115015+03:00 127004  2.980
     2 2026-07-01 03:48:20.074833+03:00  67325  1.180
     2 2026-07-01 03:49:58.434923+03:00 111512  2.600
     2 2026-07-01 03:50:24.614904+03:00  67156  1.559
     2 2026-07-01 03:54:38.314973+03:00 1

In [14]:
# Что даёт пересборка от снапшота: без неё книга накапливает расхождения после разрыва.
path, sym = SRC['bybit']
T = pd.Timestamp('2026-07-01 21:00:00+03')   # здесь внутри окна есть снапшоты

for label, kw in [('resync=False (один сид)', {'resync': False}),
                  ('resync=True (от каждого снапшота)', {'resync': True})]:
    out = build_book(path, sym, T, T + pd.Timedelta(minutes=120),
                     gap_thr=GAP_THR['bybit'], **kw)
    df = tops_frame(out)
    by_h = df.groupby(df['t'].dt.floor('30min'))['crossed'].mean()
    print(label, '-> crossed на концах пачек по получасам:',
          {str(k)[11:16]: f'{val:.2%}' for k, val in by_h.items()},
          '| пересборок:', out['stats']['resyncs'],
          '| разрывов:', out['stats']['gap'])

resync=False (один сид) -> crossed на концах пачек по получасам: {'21:00': '14.03%', '21:30': '0.93%', '22:00': '0.74%', '22:30': '1.98%'} | пересборок: 0 | разрывов: 72


resync=True (от каждого снапшота) -> crossed на концах пачек по получасам: {'21:00': '0.82%', '21:30': '0.93%', '22:00': '0.74%', '22:30': '0.83%'} | пересборок: 4 | разрывов: 68


In [15]:
# Механика двух флагов: гэп базового подфида блокирует выдачу, вторичного — только помечает.
for name in BOOKS:
    st = BOOKS[name]['stats']
    pk = max(st['packets'], 1)
    print(f"{name:<8} разрывов {st['gap']:>3}  сбросов {st['reset']:>2}  "
          f"заблокировано (feed1) {st['unhealthy_out']:>7} из {pk:>8} ({st['unhealthy_out']/pk:>6.1%})  "
          f"помечено degraded (feed2) {st['degraded_out']:>7} ({st['degraded_out']/pk:>6.1%})")


bybit    разрывов  14  сбросов  0  заблокировано (feed1)  209897 из   483339 ( 43.4%)  помечено degraded (feed2)  399785 ( 82.7%)
binance  разрывов   0  сбросов  0  заблокировано (feed1)       0 из  1543357 (  0.0%)  помечено degraded (feed2)       0 (  0.0%)
okx      разрывов   0  сбросов  0  заблокировано (feed1)       0 из   792877 (  0.0%)  помечено degraded (feed2)       0 (  0.0%)


### Пункт 2 — Flag impossible books

Crossed — это $bid > ask$, locked — равенство лучших котировок $bid = ask$, stale — пауза между публикуемыми состояниями дольше $\tau$. Считаем доли по бирже и раскладываем по часам; отдельно измеряем ортогональную величину: какая доля crossed остаётся на концах пачек, то есть видна снаружи как наблюдаемое состояние, а какая лежит внутри пачки и наружу не выходит. Порог устаревания $\tau = 250$ мс: пауза между публикуемыми состояниями у активного перпетуала измеряется десятками миллисекунд, и 5-секундный порог пропустил бы почти всё.

Проверяем и размер книги: не распухла ли она глубже снапшотной базы; этот разбор — ниже.


In [16]:
rows = []
for name in BOOKS:
    df = tops_frame(BOOKS[name])
    for hr, g in df.groupby(df['t'].dt.floor('h')):
        dt = g['t'].diff().dt.total_seconds()
        rows.append(dict(venue=name, hour=str(hr)[11:16], n=len(g),
                         crossed=float(g['crossed'].mean()),
                         locked=float(((g['bid'] == g['ask']) & g['bid'].notna()).mean()),
                         degraded=float(g['degraded'].mean()),
                         p95_gap_ms=round(float(dt.quantile(0.95))*1000, 1),
                         stale=int((dt > STALE_TAU).sum())))
per_hour = pd.DataFrame(rows)
print(per_hour.to_string(index=False))
print()
print('сводно по бирже:')
print(per_hour.groupby('venue')[['crossed','locked','degraded','stale']]
      .agg({'crossed':'mean','locked':'mean','degraded':'mean','stale':'sum'})
      .rename(columns={'crossed':'crossed_share','locked':'locked_share',
                       'degraded':'degraded_share','stale':'stale_intervals'})
      .to_string())

  venue  hour      n  crossed   locked  degraded  p95_gap_ms  stale
  bybit 09:00  95414 0.006613 0.001656  0.152357        43.0     57
  bybit 10:00 185606 0.007176 0.001778  1.000000        59.6    133
  bybit 11:00 192009 0.009687 0.001797  1.000000        53.8    157
  bybit 12:00   7633 0.009171 0.001703  1.000000        39.9      0
binance 09:00 313075 0.080859 0.000549  0.000000        26.4      1
binance 10:00 562376 0.012835 0.000629  0.000000        27.9      6
binance 11:00 617179 0.016033 0.000928  0.000000        27.8      2
binance 12:00  25773 0.000349 0.000621  0.000000        26.0      0
    okx 09:00 149797 0.001162 0.002897  0.000000        21.7      0
    okx 10:00 300421 0.000862 0.002710  0.000000        30.0      3
    okx 11:00 311126 0.001032 0.002761  0.000000        30.0      9
    okx 12:00  11493 0.000783 0.002958  0.000000        20.2      0

сводно по бирже:
         crossed_share  locked_share  degraded_share  stale_intervals
venue                       

In [17]:
# Crossed на концах пачек против crossed внутри пачек: сколько из них вообще видно снаружи.
for name in BOOKS:
    st = BOOKS[name]['stats']
    print(f"{name:<8} конец пачки: {st['packets_crossed']:>6} из {st['packets']:>7} "
          f"({st['packets_crossed']/st['packets']:.4%})   "
          f"внутри пачки: {st['crossed_rows']:>7} из {st['rows']:>9} ({st['crossed_rows']/st['rows']:.4%})")

bybit    конец пачки:   3913 из  483339 (0.8096%)   внутри пачки:   83981 из   2803402 (2.9957%)
binance  конец пачки:  42445 из 1543357 (2.7502%)   внутри пачки:  524511 из  16941964 (3.0959%)
okx      конец пачки:    793 из  792877 (0.1000%)   внутри пачки:   11967 из  10461646 (0.1144%)


In [18]:
# Вклад подфида удалений: без feed3 книга внутри пачки склеена заметно хуже.
print('feed 1+2 против 1+2+3 (первые 20 минут окна):')
for name, (path, sym) in SRC.items():
    t_end = W0 + pd.Timedelta(minutes=20)
    fracs = {}
    for feeds in [(1, 2), (1, 2, 3)]:
        b, a, sq, st = seed_book(path, sym, W0)
        o = reconstruct(iter_batches(path, sym, st, t_end, feeds=feeds), b, a, sq,
                        seq_feeds=(1, 2), t_record_from=W0, t_end=t_end,
                        gap_thr=GAP_THR[name], resync=[])
        s = o['stats']
        fracs[feeds] = (s['crossed_rows'] / max(s['rows'], 1),
                        s['packets_crossed'] / max(s['packets'], 1))
    print(f"{name:<8} feed1+2: внутри {fracs[(1,2)][0]:.2%}, на концах {fracs[(1,2)][1]:.2%}   "
          f"feed1+2+3: внутри {fracs[(1,2,3)][0]:.2%}, на концах {fracs[(1,2,3)][1]:.2%}")

feed 1+2 против 1+2+3 (первые 20 минут окна):


bybit    feed1+2: внутри 99.80%, на концах 99.74%   feed1+2+3: внутри 2.83%, на концах 0.76%


binance  feed1+2: внутри 99.55%, на концах 99.44%   feed1+2+3: внутри 10.70%, на концах 9.22%


okx      feed1+2: внутри 99.78%, на концах 99.70%   feed1+2+3: внутри 0.15%, на концах 0.14%


### Глубина, которой не было в снапшоте

Книга бывает невозможна и по размеру: восстановленный стакан не должен заметно превышать глубину, которая вообще приходила снапшотом. Снапшот задаёт базу — Bybit 50+50 уровней, Binance 100+100, OKX 400+400, — и за её пределами у сборщика нет ни исходных цен, ни снятий: дельты добавляют новые уровни, но снятия для цен, которых не было в базе, в дамп могут не прийти, и книга распухает фантомными уровнями. Сравним глубину снапшота с глубиной восстановленного стакана.


In [19]:
# Уровни в первом снапшоте против уровней восстановленного стакана.
for name, (path, sym) in SRC.items():
    snap_t = con.execute("""
        SELECT min(event_time) FROM read_parquet(?)
        WHERE symbol=? AND _feed=1 AND entry_type=1""", [path, sym]).fetchone()[0]
    sn = con.execute("""
        SELECT count(*) n, count(*) FILTER (WHERE side=1) nb, count(*) FILTER (WHERE side=-1) na
        FROM read_parquet(?) WHERE symbol=? AND _feed=1 AND entry_type=1 AND event_time=?""",
        [path, sym, snap_t]).fetchone()
    st = BOOKS[name]['stats']
    pk = max(st['packets'], 1)
    print(f"{name:<8} снапшот {int(sn[0]):>4} строк ({int(sn[1])} bid / {int(sn[2])} ask) | "
          f"восстановлено уровней в среднем {st['sum_bid_levels']//pk}/{st['sum_ask_levels']//pk}, "
          f"максимум {st['max_bid_levels']}/{st['max_ask_levels']}")


bybit    снапшот  100 строк (50 bid / 50 ask) | восстановлено уровней в среднем 49/49, максимум 52/52


binance  снапшот  200 строк (100 bid / 100 ask) | восстановлено уровней в среднем 11037/9670, максимум 14482/12015


okx      снапшот  800 строк (400 bid / 400 ask) | восстановлено уровней в среднем 399/399, максимум 402/402


### Пункт 3 — Measure feed health

Латентность — это $\ell = \big(\mathrm{epoch}(T^{recv}) + n^{recv}/10^9\big) - \big(\mathrm{epoch}(T^{exch}) + n^{exch}/10^9\big)$ с наносекундной точностью приёма: `event_time` усечён до миллисекунд, а микросекунды лежат в `event_time_nanos`. Считаем её по подфидам на полном периоде, затем сравниваем медианы бирж между собой — так получается кросс-биржевой сдвиг; в него входит и реальное время доставки, поэтому к расхождению часов он относится лишь отчасти. Чтобы отделить смещение часов от сетевого пути, добавляем вторую, независимую оценку: кросс-корреляцию приростов mid между биржами по 100-мс сетке. Если часы разъехались, приросты коррелируют сильнее при ненулевом лаге.

Отдельно ищем случай, когда порядок `exchange_time` противоречит порядку прихода: внутри одного `event_time` наша книга вынуждена применять несколько сообщений, и упорядочивание по биржевому времени дало бы другую книгу.

In [20]:
lat_tables = {}
for name, (path, sym) in SRC.items():
    tic = time.time()
    lat_tables[name] = con.execute("""
        SELECT _feed, count(*) n,
               round(quantile_cont(epoch(event_time)+event_time_nanos/1e9
                     - epoch(exchange_time)-exchange_time_nanos/1e9, 0.50),5) p50,
               round(quantile_cont(epoch(event_time)+event_time_nanos/1e9
                     - epoch(exchange_time)-exchange_time_nanos/1e9, 0.95),5) p95,
               round(quantile_cont(epoch(event_time)+event_time_nanos/1e9
                     - epoch(exchange_time)-exchange_time_nanos/1e9, 0.99),5) p99,
               round(quantile_cont(epoch(event_time)+event_time_nanos/1e9
                     - epoch(exchange_time)-exchange_time_nanos/1e9, 0.999),4) p999,
               round(max(epoch(event_time)+event_time_nanos/1e9
                     - epoch(exchange_time)-exchange_time_nanos/1e9),3) max,
               round(min(epoch(event_time)+event_time_nanos/1e9
                     - epoch(exchange_time)-exchange_time_nanos/1e9),3) min
        FROM read_parquet(?) WHERE symbol=? AND exchange_time IS NOT NULL
        GROUP BY 1 ORDER BY 1""", [path, sym]).df()
    print(name, f'({time.time()-tic:.0f}s)')
    print(lat_tables[name].to_string(index=False))


bybit (11s)
 _feed        n     p50     p95     p99   p999   max   min
     1 57746425 0.03729 0.04004 0.04650 0.0657 0.391 0.035
     2  4484944 0.03689 0.03985 0.04415 0.0554 0.196 0.035
     3   943667 0.04022 0.04429 0.04967 0.0634 0.095 0.037


binance (83s)
 _feed         n     p50     p95     p99   p999   max   min
     1 288587910 0.00229 0.00406 0.00500 0.0485 0.168 0.002
     2  78538820 0.00210 0.00274 0.00494 0.0265 0.175 0.001
     3   2199381 0.00200 0.00273 0.00386 0.0087 0.134 0.001
     4      5628 0.00673 0.05524 0.10059 0.1104 0.113 0.003


okx (33s)
 _feed         n     p50     p95     p99   p999   max   min
     1 202731500 0.02393 0.02663 0.03096 0.0740 2.581 0.021
     2   4464008 0.02361 0.02449 0.02567 0.0378 0.812 0.021
     3   2024198 0.02489 0.02972 0.04046 0.2122 0.850 0.023


In [21]:
# Кросс-биржевой сдвиг: медиана задержки биржи есть смещение её часов к нашим.
med = {}
for name in lat_tables:
    t = lat_tables[name]
    t = t[t['_feed'].isin([1, 2])]
    med[name] = float((t['n'] * t['p50']).sum() / t['n'].sum())
print('медианная задержка, мс:', {k: round(v*1000, 2) for k, v in med.items()})
base = min(med, key=med.get)
print('сдвиг относительно', base, 'мс:', {k: round((v-med[base])*1000, 2) for k, v in med.items()})
print('размах между биржами: %.2f мс' % ((max(med.values())-min(med.values()))*1000))

медианная задержка, мс: {'bybit': 37.26, 'binance': 2.25, 'okx': 23.92}
сдвиг относительно binance мс: {'bybit': 35.01, 'binance': 0.0, 'okx': 21.67}
размах между биржами: 35.01 мс


In [22]:
# Вторая оценка сдвига: кросс-корреляция приростов mid между биржами по 100-мс сетке.
piv = {}
for ex in PAIR:
    piv[ex] = ref[ref.exchange == ex].set_index('event_time')['mid'].resample('100ms').last().ffill()
M = pd.DataFrame(piv).dropna()
dM = M.diff().dropna()
print('сетка', len(M), M.index[0], '->', M.index[-1])

def best_lag(x, y, lags=range(-5, 6)):
    # x сдвигается вперёд на lag тактов; ищем лаг максимума корреляции.
    out = []
    for lag in lags:
        a = x[lag:] if lag >= 0 else x[:lag]
        b = y[:len(y)-lag] if lag >= 0 else y[-lag:]
        out.append((lag, float(np.corrcoef(a, b)[0, 1])))
    return max(out, key=lambda z: z[1]), float(np.corrcoef(x, y)[0, 1])

for a, b in [('Binance','Bybit'), ('Binance','Okex'), ('Bybit','Okex')]:
    (lag, corr), corr0 = best_lag(dM[a].to_numpy(), dM[b].to_numpy())
    print(f'{a} vs {b}: максимум при лаге {lag} x100 мс (corr {corr:.3f}), на нулевом лаге {corr0:.3f}')

сетка 90000 2026-07-01 09:32:00+03:00 -> 2026-07-01 12:01:59.900000+03:00
Binance vs Bybit: максимум при лаге -1 x100 мс (corr 0.620), на нулевом лаге 0.355
Binance vs Okex: максимум при лаге 0 x100 мс (corr 0.545), на нулевом лаге 0.545
Bybit vs Okex: максимум при лаге 0 x100 мс (corr 0.501), на нулевом лаге 0.501


In [23]:
# Ми-сортировка: наш receive-time склеивает много биржевых апдейтов в один момент,
# а exchange_time внутри этого момента разрежает их по-своему.
for name, (path, sym) in SRC.items():
    df = con.execute("""
        SELECT event_time, side, price FROM read_parquet(?, file_row_number=true)
        WHERE symbol=? AND entry_type=2 AND side IN (1,-1) AND amount>0
          AND event_time>? AND event_time<=?
        ORDER BY event_time, file_row_number""", [path, sym, W0, W0+pd.Timedelta(minutes=5)]).df()
    by_t = df.groupby('event_time').size()
    overlap = 0
    for t, g in df.groupby('event_time'):
        mb = g.loc[g.side == 1, 'price'].max()
        ma = g.loc[g.side == -1, 'price'].min()
        if np.isfinite(mb) and np.isfinite(ma) and mb >= ma:
            overlap += 1
    print(f"{name:<8} строк на event_time: медиана {by_t.median():.0f}, p99 {by_t.quantile(0.99):.0f}, "
          f"max {by_t.max()}; event_time с bid/ask на перекрывающихся ценах: {overlap}")

bybit    строк на event_time: медиана 2, p99 49, max 100; event_time с bid/ask на перекрывающихся ценах: 0


binance  строк на event_time: медиана 1, p99 151, max 964; event_time с bid/ask на перекрывающихся ценах: 49


okx      строк на event_time: медиана 7, p99 109, max 707; event_time с bid/ask на перекрывающихся ценах: 2


### Порядок приёма против порядка биржи

Порядок по биржевому времени внутри одной пачки мог бы отличаться от порядка приёма; проверяем это напрямую. Порядок приёма берём с полной точностью — `event_time + event_time_nanos`, как в латентности пункта 3, — и считаем, сколько раз биржевой штамп (`exchange_time + exchange_time_nanos`) убывает вдоль этого порядка. Отдельно проверим, ненулевой ли вообще `exchange_time_nanos`: если он нулевой, сравниваем наносекундный приём против миллисекундной биржи.


In [24]:
# Инверсии порядка: сколько раз exchange_time идёт назад вдоль порядка приёма.
for name, (path, sym) in SRC.items():
    tic = time.time()
    df = con.execute("""
        WITH p AS (
          SELECT epoch(event_time) + event_time_nanos/1e9 AS recv,
                 epoch(exchange_time) + exchange_time_nanos/1e9 AS exch,
                 _feed, min(file_row_number) frn
          FROM read_parquet(?, file_row_number=true)
          WHERE symbol=? AND entry_type=2 AND event_time>? AND event_time<=?
          GROUP BY 1,2,3),
        s AS (SELECT recv, exch, row_number() OVER (ORDER BY recv, frn) rn FROM p),
        d AS (SELECT recv, exch, exch - lag(exch) OVER (ORDER BY rn) dd FROM s)
        SELECT count(*) n_pkt,
               sum(CASE WHEN dd < 0 THEN 1 ELSE 0 END) inversions,
               min(dd) worst
        FROM d""", [path, sym, W0, W1]).df().iloc[0]
    nz = con.execute("""
        SELECT sum(CASE WHEN exchange_time_nanos <> 0 THEN 1 ELSE 0 END) ex_nz,
               sum(CASE WHEN event_time_nanos <> 0 THEN 1 ELSE 0 END) ev_nz, count(*) n
        FROM read_parquet(?) WHERE symbol=? AND entry_type=2 AND event_time>? AND event_time<=?""",
        [path, sym, W0, W1]).fetchone()
    print(f"{name:<8} пакетов {int(df.n_pkt):>9}  инверсий порядка(приём) vs порядок(биржа) "
          f"{int(df.inversions):>6} ({df.inversions/df.n_pkt:.3%})  худший шаг {df.worst:+.3f} с  "
          f"({time.time()-tic:.0f}s)")
    print(f"         ненулевой exchange_time_nanos: {int(nz[0] or 0)} из {int(nz[2]):,}; "
          f"event_time_nanos: {int(nz[1] or 0):,} из {int(nz[2]):,}")


bybit    пакетов    480981  инверсий порядка(приём) vs порядок(биржа)    875 (0.182%)  худший шаг -0.031 с  (0s)
         ненулевой exchange_time_nanos: 0 из 2,788,792; event_time_nanos: 2,785,997 из 2,788,792


binance  пакетов   1622912  инверсий порядка(приём) vs порядок(биржа)   3533 (0.218%)  худший шаг -0.103 с  (1s)
         ненулевой exchange_time_nanos: 0 из 16,722,047; event_time_nanos: 16,706,537 из 16,722,047


okx      пакетов    776364  инверсий порядка(приём) vs порядок(биржа)    628 (0.081%)  худший шаг -0.100 с  (1s)
         ненулевой exchange_time_nanos: 0 из 10,265,712; event_time_nanos: 10,255,891 из 10,265,712


### Пункт 4 — Post-trade on simulated fills

Заявленная задержка — 50 мс: решение принимается по книге в момент $t$, ордер приходит на биржу в $t+50$ мс и исполняется тейкером по книге в этот момент. Для каждого снимка сетки и каждого размера проходим лестницу уровней и получаем среднюю цену исполнения $P_{fill}$. Проход по книге ограничен 100 уровнями, снапшотной базой Bybit и Binance: за пределами снапшота восстановленный стакан Binance раздут фантомными уровнями (пункт 2), и опираться на такую глубину нельзя. Slippage считаем и к decision mid $M_{dec}$ (сигнал), и к arrival mid $M_{arr}$ (приход): $s = \mathrm{side}\cdot(P_{fill}-M)/M$. Разница между ними показывает дрейф середины за время задержки. На размерах до 1 BTC весь объём укладывается в один ценовой уровень, и медиана упирается в половину тика: цену квантует тик-сетка; настоящая стоимость появляется, когда объём задевает несколько уровней.

Отдельно помечаем partial fills: если в снимке глубины не хватило, чтобы набрать объём, средняя цена считается по неполному проходу и теряет смысл — для покупки она уезжает ниже середины, давая физически невозможный отрицательный slippage. Такие точки исключаем из медиан и показываем их долю.

Размеры берём 0.1, 0.5, 1, 5, 20, 50 BTC — от $\approx$6 тыс. до $\approx$3 млн номинала; влияние самих сделок на книгу не моделируется, поэтому это верхняя граница издержек.

In [25]:
DEPTH_L = 100          # уровней в снимке; проход по книге обрезаем до снапшотной базы
TICK = 0.1             # шаг цены BTCUSDT
FILL_SIZES = [0.1, 0.5, 1, 5, 20, 50]
LAG_MS = 50
HORIZONS_MS = [100, 1000, 10000]


def depth_arrays(depths, L=DEPTH_L):
    n = len(depths)
    bid_px = np.full((n, L), np.nan); bid_q = np.zeros((n, L))
    ask_px = np.full((n, L), np.nan); ask_q = np.zeros((n, L))
    for i, (t, b, a) in enumerate(depths):
        for j, (p, q) in enumerate(b[:L]): bid_px[i, j] = p; bid_q[i, j] = q
        for j, (p, q) in enumerate(a[:L]): ask_px[i, j] = p; ask_q[i, j] = q
    return (np.array([d[0] for d in depths]), bid_px, bid_q, ask_px, ask_q)


def walk(px, q, size):
    # Средняя цена прохода и доля объёма, которую удалось набрать.
    c = np.cumsum(q)
    avail = c[-1]
    if avail <= 0:
        return np.nan, 0.0
    k = int(np.searchsorted(c, size))
    if k < len(px) and np.isfinite(px[k]) and avail >= size:
        prev = c[k-1] if k > 0 else 0.0
        return float((np.nansum(px[:k]*q[:k]) + px[k]*(size - prev)) / size), 1.0
    return float(np.nansum(px*q) / avail), float(avail / size)   # неполный проход


ARRAYS = {name: depth_arrays(BOOKS[name]['depths']) for name in BOOKS}
for name, (tt, bp, bq, ap, aq) in ARRAYS.items():
    step_ms = np.median(np.diff(pd.to_datetime(tt).astype('int64')))/1e6
    print(f'{name:<8} снимков глубины {len(tt)}, шаг {step_ms:.0f} мс')

bybit    снимков глубины 192175, шаг 0 мс
binance  снимков глубины 619460, шаг 0 мс


okx      снимков глубины 311411, шаг 0 мс


In [26]:
# Решение берём из снимка в момент t, исполнение — из снимка t+50 мс, markout — из mid(t+50+h):
# ни одна величина не заглядывает вперёд, look-ahead отсутствует.
STEP = 4            # решение каждые 200 мс
LAG_I = LAG_MS // GRID_MS
MAXH_I = max(HORIZONS_MS) // GRID_MS

fills = []
for name, (tt, bp, bq, ap, aq) in ARRAYS.items():
    n = len(tt)
    mid = (bp[:,0] + ap[:,0]) / 2.0
    for i in range(0, n - LAG_I - MAXH_I, STEP):
        if not np.isfinite(mid[i]) or not np.isfinite(mid[i+LAG_I]):
            continue
        j = i + LAG_I
        for size in FILL_SIZES:
            for side, q in (('buy', 1), ('sell', -1)):
                px, qq = (ap[j], aq[j]) if side == 'buy' else (bp[j], bq[j])
                pf, filled = walk(px, qq, size)
                if not np.isfinite(pf):
                    continue
                rec = dict(venue=name, side=side, size=size, t=tt[i],
                           mid_dec=mid[i], mid_arr=mid[j], fill=pf,
                           partial=filled < 0.999)
                rec['slip_dec_bps'] = q*(pf - mid[i])/mid[i]*1e4
                rec['slip_arr_bps'] = q*(pf - mid[j])/mid[j]*1e4
                for h in HORIZONS_MS:
                    idx = i + LAG_I + h//GRID_MS
                    m = mid[idx] if idx < n else np.nan
                    rec[f'mo_{h}'] = q*(m - pf)/mid[i]*1e4 if np.isfinite(m) else np.nan
                fills.append(rec)

fills = pd.DataFrame(fills)
print('симуляций:', len(fills), '| окно глубины', ARRAYS['bybit'][0][0], '->', ARRAYS['bybit'][0][-1])
print()
print('доля partial fills (глубины не хватило до полного объёма):')
print(fills.groupby(['venue','size'])['partial'].mean().unstack('size')
      .to_string(float_format=lambda v: f'{v:.1%}'))

симуляций: 3367344 | окно глубины 2026-07-01 11:02:00.003096+03:00 -> 2026-07-01 12:01:59.983546+03:00

доля partial fills (глубины не хватило до полного объёма):


size     0.1   0.5   1.0   5.0   20.0  50.0
venue                                      
binance  0.0%  0.0%  0.0%  0.0% 25.3% 90.0%
bybit    0.0%  0.0%  0.1%  3.6% 69.8% 98.8%
okx      0.0%  0.0%  0.0%  0.0%  4.1% 67.5%


In [27]:
fills['slip_dec_tick'] = fills['slip_dec_bps'] * fills['mid_dec'] / 1e4 / TICK
fills['slip_arr_tick'] = fills['slip_arr_bps'] * fills['mid_arr'] / 1e4 / TICK
full = fills[~fills['partial']]

print('Slippage к decision mid, тики (только полные филлы), медиана и p95:')
print(full.groupby(['venue','size'])['slip_dec_tick']
      .agg(med='median', p95=lambda x: x.quantile(0.95)).round(3).unstack('size').to_string())
print()
print('Slippage к arrival mid, тики (только полные филлы), медиана и p95:')
print(full.groupby(['venue','size'])['slip_arr_tick']
      .agg(med='median', p95=lambda x: x.quantile(0.95)).round(3).unstack('size').to_string())
print()
print('Cost vs size (arrival mid), полные филлы: медианные тики, USD на ордер и p95 тиков:')
tab = full.groupby(['venue','size']).agg(
      ticks=('slip_arr_tick','median'),
      usd=('slip_arr_tick', lambda x: (x.median()*TICK*full.loc[x.index,'size']).median()),
      p95_ticks=('slip_arr_tick', lambda x: x.quantile(0.95))).round(3)
print(tab.unstack('size').to_string())

Slippage к decision mid, тики (только полные филлы), медиана и p95:


         med                                    p95                                        
size    0.1  0.5  1.0    5.0     20.0    50.0  0.1     0.5     1.0     5.0     20.0    50.0
venue                                                                                      
binance  0.5  0.5  0.5  0.500  26.636  20.993  1.53  13.546  24.528  53.769  69.626  59.631
bybit    0.5  0.5  0.5  7.825  21.650  18.482  3.47  15.226  24.312  50.129  45.736  66.017
okx      0.5  0.5  0.5  0.500  37.506  58.709  0.50  12.668  22.307  50.876  87.046  87.179

Slippage к arrival mid, тики (только полные филлы), медиана и p95:


         med                                    p95                                        
size    0.1  0.5  1.0    5.0     20.0    50.0  0.1     0.5     1.0     5.0     20.0    50.0
venue                                                                                      
binance  0.5  0.5  0.5  0.500  26.745  21.153  1.69  13.286  24.237  53.658  69.424  60.028
bybit    0.5  0.5  0.5  8.099  22.005  22.753  2.99  13.978  22.979  49.540  46.004  67.133
okx      0.5  0.5  0.5  0.500  37.536  58.822  0.50  12.000  21.278  50.305  86.638  87.163

Cost vs size (arrival mid), полные филлы: медианные тики, USD на ордер и p95 тиков:


        ticks                                     usd                                      p95_ticks                                        
size     0.1  0.5  1.0    5.0     20.0    50.0   0.1    0.5   1.0    5.0     20.0     50.0      0.1     0.5     1.0     5.0     20.0    50.0
venue                                                                                                                                       
binance   0.5  0.5  0.5  0.500  26.745  21.153  0.005  0.025  0.05  0.250  53.489  105.764      1.69  13.286  24.237  53.658  69.424  60.028
bybit     0.5  0.5  0.5  8.099  22.005  22.753  0.005  0.025  0.05  4.049  44.010  113.763      2.99  13.978  22.979  49.540  46.004  67.133
okx       0.5  0.5  0.5  0.500  37.536  58.822  0.005  0.025  0.05  0.250  75.072  294.108      0.50  12.000  21.278  50.305  86.638  87.163


In [28]:
# Markouts в базисных пунктах: на 100 мс эффект одного тика около 0.017 б.п., точность нужна до тысячных.
for h in HORIZONS_MS:
    f = full.assign(pos=(full[f'mo_{h}'] > 0).astype(float))
    g = f.groupby(['venue','side']).agg(med=(f'mo_{h}','median'), mean=(f'mo_{h}','mean'),
                                       pos=('pos','mean'))
    allm = f.groupby('venue')[f'mo_{h}'].mean()
    print(f'markout {h:>6} мс, б.п. (медиана / среднее / доля > 0):')
    print(g.to_string(float_format=lambda v: f'{v:+.4f}'))
    print('среднее по обеим сторонам:', {k: round(v, 4) for k, v in allm.items()})
    print()

markout    100 мс, б.п. (медиана / среднее / доля > 0):
                 med    mean     pos
venue   side                        
binance buy  -0.0085 -0.1401 +0.0147
        sell -0.0085 -0.1327 +0.0163
bybit   buy  -0.0085 -0.1270 +0.0080
        sell -0.0085 -0.1046 +0.0082
okx     buy  -0.0085 -0.2319 +0.0040
        sell -0.0085 -0.2346 +0.0033
среднее по обеим сторонам: {'binance': -0.1364, 'bybit': -0.1158, 'okx': -0.2332}



markout   1000 мс, б.п. (медиана / среднее / доля > 0):
                 med    mean     pos
venue   side                        
binance buy  -0.0085 -0.1423 +0.0498
        sell -0.0085 -0.1375 +0.0482
bybit   buy  -0.0085 -0.1329 +0.0891
        sell -0.0085 -0.1177 +0.0918
okx     buy  -0.0085 -0.2337 +0.0457
        sell -0.0085 -0.2410 +0.0452
среднее по обеим сторонам: {'binance': -0.1399, 'bybit': -0.1253, 'okx': -0.2373}



markout  10000 мс, б.п. (медиана / среднее / доля > 0):
                 med    mean     pos
venue   side                        
binance buy  -0.0085 -0.1511 +0.2507
        sell -0.0085 -0.1705 +0.2276
bybit   buy  -0.0085 -0.1044 +0.4059
        sell -0.0472 -0.1622 +0.3730
okx     buy  -0.0141 -0.2211 +0.2743
        sell -0.0426 -0.2752 +0.2586
среднее по обеим сторонам: {'binance': -0.1607, 'bybit': -0.1333, 'okx': -0.248}



Медиана markout одна и та же на обеих сторонах: −0.0085 б.п. там, где исполнение укладывается в лучший уровень. Совпадение закономерно: обе стороны упираются в полтину спреда. Медиана slippage — ровно половина тика, и markout на 100 мс меряет ту же величину с обратным знаком. Медиана слепа к асимметрии, поэтому смотрим средние по сторонам.

Среднее по всем заказам почти не движется с горизонтом и близко у трёх бирж. Причина в симметричной выборке: в каждой точке решения есть и покупка, и продажа против одного и того же $M(t_{fill}+\delta)$, их вклады в общее среднее сокращаются, и от пары остаётся $\tfrac{1}{2}(\bar P_{sell}-\bar P_{buy})/M_d$ — величина, от горизонта не зависящая, потому что $\bar P$ берётся из стакана в $t+50$ мс. Содержательны именно средние по сторонам: они показывают, уходит ли цена против нашей стороны после исполнения. Разница между покупкой и продажей — слабый признак adverse selection, но на сотне точек решения она статистически не значима, и читать её как устойчивый эффект нельзя.


### Пункт 5 — Inject & survive

Берём пятиминутный кусок фида Bybit сразу после снапшота (09:31:05, до следующего снапшота далеко, поэтому на чистом куске книга не пересобирается и разрывы не маскируются). Режем его на сообщения (блоки) и портим копии четырьмя способами:

- drop — выбрасываем 500 подряд идущих сообщений feed1: номер уезжает вперёд;
- duplicate — переотправляем старое сообщение feed1 позже: номер идёт назад, но уже виден;
- reorder — меняем местами два соседних сообщения feed1: номер идёт назад, и этот номер ещё не встречался (сброс);
- reset — сбрасываем номер у хвоста feed1 на −1: сброс в новое пространство нумерации.

Прогоняем каждую копию через тот же сборщик. За результат берём отклонение от чистого прогона, а не абсолютное число нарушений: часть инвариантов (например, редкость crossed на концах пачек) нарушается и на живом фиде, и без базы результат лишён смысла.

In [29]:
def load_blocks(path, sym, t0, t1):
    res = con.execute(
        "SELECT event_time t, _feed feed, sequence_no seq, side, price, amount "
        "FROM read_parquet(?, file_row_number=true) "
        "WHERE symbol=? AND entry_type=2 AND _feed IN (1,2,3) AND event_time>? AND event_time<=? "
        "ORDER BY event_time, file_row_number", [path, sym, t0, t1])
    blocks, cur_key, cur = [], None, None
    while True:
        chunk = res.fetch_df_chunk(20)
        if chunk.shape[0] == 0:
            break
        for t, f, seq, s, p, a in chunk.itertuples(index=False):
            q = None if (seq is None or type(seq).__name__ == 'NAType') else int(seq)
            key = (f, q if f in (1, 2) else t)
            if key != cur_key:
                cur = [f, t, q, []]
                blocks.append(cur); cur_key = key
            cur[3].append((s, p, a))
    return blocks


def copy_blocks(blocks):
    return [[b[0], b[1], b[2], list(b[3])] for b in blocks]


def feed1_idx(blocks):
    return [i for i, b in enumerate(blocks) if b[0] == 1]


def inj_drop(blocks):          # теряем 500 сообщений подряд -> шаг sequence_no больше порога
    drop = set(feed1_idx(blocks)[300:800])
    return [b for i, b in enumerate(copy_blocks(blocks)) if i not in drop]


def inj_duplicate(blocks):     # переотправляем старое сообщение позже -> номер назад, уже виденный
    b = copy_blocks(blocks)
    f1 = feed1_idx(blocks)
    b.insert(f1[500], copy_blocks([blocks[f1[200]]])[0])
    return b


def inj_reorder(blocks):       # меняем местами два соседних сообщения feed1 -> номер назад, новый
    b = copy_blocks(blocks)
    f1 = feed1_idx(blocks)
    i, j = f1[400], f1[401]
    b[i], b[j] = b[j], b[i]
    return b


def inj_reset(blocks):         # сбрасываем номер вниз до невиденного
    b = copy_blocks(blocks)
    for i in feed1_idx(blocks)[600:]:
        b[i][2] = -1
    return b


def iter_blocks(blocks, batch=5000):
    t_l, f_l, q_l, s_l, p_l, a_l = [], [], [], [], [], []
    for f, t, seq, ev in blocks:
        for s, p, a in ev:
            t_l.append(t); f_l.append(f); q_l.append(seq)
            s_l.append(s); p_l.append(p); a_l.append(a)
            if len(t_l) >= batch:
                yield (t_l, f_l, q_l, s_l, p_l, a_l)
                t_l, f_l, q_l, s_l, p_l, a_l = [], [], [], [], [], []
    if t_l:
        yield (t_l, f_l, q_l, s_l, p_l, a_l)


def replay(blocks, path, sym, gap_thr, resync=None):
    bids, asks, snap_seq, _ = seed_book(path, sym, blocks[0][1])
    return reconstruct(iter_blocks(blocks), bids, asks, snap_seq, seq_feeds=(1, 2),
                       t_record_from=blocks[0][1],
                       t_end=blocks[-1][1] + pd.Timedelta(seconds=1),
                       gap_thr=gap_thr, resync=resync or [])

In [30]:
path, sym = SRC['bybit']
BT0 = pd.Timestamp('2026-07-01 09:31:05.480956+03')     # первый снапшот Bybit
BT1 = BT0 + pd.Timedelta(minutes=5)
base_blocks = load_blocks(path, sym, BT0, BT1)
print('блоков в куске:', len(base_blocks), '| старт', BT0)

cases = {'clean': copy_blocks(base_blocks), 'drop': inj_drop(base_blocks),
         'duplicate': inj_duplicate(base_blocks), 'reorder': inj_reorder(base_blocks),
         'reset': inj_reset(base_blocks)}

sum_rows = []
for label, bl in cases.items():
    out = replay(bl, path, sym, GAP_THR['bybit'])
    df = tops_frame(out); st = out['stats']
    dt = df['t'].diff().dt.total_seconds().dropna()
    sum_rows.append(dict(case=label, packets=st['packets'],
                         crossed=st['packets_crossed']/max(st['packets'],1),
                         gaps=st['gap'], resets=st['reset'], dups=st['dup_msg'],
                         chronology=bool((dt >= 0).all())))

sm = pd.DataFrame(sum_rows).set_index('case')
print(sm.to_string(float_format=lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)))
print()
base_row = sm.loc['clean']
delta = pd.DataFrame({
    'crossed_delta': sm['crossed'] - base_row['crossed'],
    'gaps_delta':    sm['gaps'] - base_row['gaps'],
    'resets_delta':  sm['resets'] - base_row['resets'],
    'dups_delta':    sm['dups'] - base_row['dups'],
})
print('отклонение от чистого прогона:')
print(delta.to_string(float_format=lambda v: f'{v:+.4f}' if isinstance(v, float) else str(v)))

блоков в куске: 17683 | старт 2026-07-01 09:31:05.480956+03:00


           packets  crossed  gaps  resets  dups  chronology
case                                                       
clean        17670   0.0096     0       0     0        True
drop         17171   0.3899     1       0     0        True
duplicate    17671   0.0403     1       0     1       False
reorder      17670   0.0096     0       1     0       False
reset        17670   0.0096     0       1     0        True

отклонение от чистого прогона:
           crossed_delta  gaps_delta  resets_delta  dups_delta
case                                                          
clean            +0.0000           0             0           0
drop             +0.3803           1             0           0
duplicate        +0.0307           1             0           1
reorder          +0.0000           0             1           0
reset            +0.0000           0             1           0


In [31]:
# Итоговый набор: инвариант нарушен, если отклонение от чистого прогона значимо.
CROSSED_TOL = 0.02     # 2 п.п. — заведомо выше шума чистого прогона (~1 %)
battery = []
for case in sm.index:
    d = delta.loc[case]
    battery.append(dict(
        case=case,
        chronology=bool(sm.loc[case, 'chronology']),
        no_crossed_at_packet_end=bool(d['crossed_delta'] <= CROSSED_TOL),
        no_unrecovered_gap=bool(d['gaps_delta'] <= 0),
        no_sequence_reset=bool(d['resets_delta'] <= 0),
        no_duplicate_message=bool(d['dups_delta'] <= 0)))
bat = pd.DataFrame(battery).set_index('case')
print(bat.to_string())
print()
caught = {c: [inv for inv in bat.columns if not bat.loc[c, inv]] for c in bat.index}
print('какие инварианты падают по случаям:')
for c, invs in caught.items():
    print(f'  {c:<10} -> {invs if invs else "все инварианты проходят"}')
print()
print('чистый прогон проходит все инварианты:', bool(bat.loc['clean'].all()))
print('каждый из четырёх инъекций ловится хотя бы одним инвариантом:',
      all(len(caught[c]) > 0 for c in ['drop','duplicate','reorder','reset']))

           chronology  no_crossed_at_packet_end  no_unrecovered_gap  no_sequence_reset  no_duplicate_message
case                                                                                                        
clean            True                      True                True               True                  True
drop             True                     False               False               True                  True
duplicate       False                     False               False               True                 False
reorder         False                      True                True              False                  True
reset            True                      True                True              False                  True

какие инварианты падают по случаям:
  clean      -> все инварианты проходят
  drop       -> ['no_crossed_at_packet_end', 'no_unrecovered_gap']
  duplicate  -> ['chronology', 'no_crossed_at_packet_end', 'no_unrecovered_gap', 'no_duplicat

## Выводы и ограничения

Данные трёх бирж почти чистые по последовательности. Порог разрыва откалиброван по данным, а не назначен: для каждого подфида это максимальный шаг `sequence_no`, наблюдённый при паузе приёма до секунды, плюс единица. У Bybit это 43.8/59.7 тыс. для feed1/feed2, у Binance 55.4/113.8 тыс., у OKX 10.4/63.0 тыс. — разброс в разы объясняет, почему единый порог в 100 тыс. был бы неверен. Второй признак — пауза приёма больше секунды — отсекает толстые дельты: без него одного высокого шага недостаточно, чтобы отличить потерю от плотной дельты. За полный период калиброванная детекция даёт 336 разрывов у Bybit (8 в feed1 и 328 в feed2), 2 у Binance, 1 у OKX — и ни одного шага назад. В 2.5-часовом окне гэпов 14 у Bybit и ни одного у Binance с OKX; на языке последствий это один блокирующий гэп базового feed1 и тринадцать помечающих во вторичном feed2.

Пересборка от снапшота — то, на чём держится книга. В окне 21:00–23:00 без пересборки доля crossed на концах пачек в первом получасе доходит до 14.0 %, а с пересборкой от четырёх снапшотов падает до 0.8 %. Ещё важнее то, чего без пересборки не видно: сброс нумерации. Номер снапшота Bybit оказывается ниже последнего номера инкрементов на тысячи (09:31: −8836, 21:10: −12204, 22:02: −2834, 22:50: −642, 22:58: −2643) — поток переоткрывается, и сборщик без пересборки счёл бы это непрерывным разрывом.

Два флага разведены не для красоты, а по последствиям. Блокирует выдачу только гэп базового feed1: он несёт базу и основной поток изменений, и до снапшота потерянные обновления неотличимы от разреженного счётчика. В 2.5-часовом окне у Bybit такой гэп один, и он не закрыт снапшотом до конца окна, поэтому 43.4 % состояний помечены непригодными. Гэпы вторичного feed2 искажают глубину, но не best bid/ask, и выдачу не останавливают: их в окне 13, они помечают 82.7 % состояний меткой degraded, и эти состояния продолжают выдаваться. Независимая проверка «не выдавать стакан через разрыв» согласуется с флагом: внутри интервала «гэп feed1 → ближайший снапшот» пригодных состояний 0. Это честнее, чем отдавать рассинхронизированный стакан; на живом потоке снапшот пришёл бы раньше и заморозка была бы короче.

Crossed-состояния почти целиком живут внутри пачки. На концах пачек доля crossed — 0.81 % у Bybit, 2.75 % у Binance (первые полчаса после старта — до 8 %, дальше 1.3–1.6 %) и 0.10 % у OKX; внутри пачки — 3.0 %, 3.1 % и 0.11 % соответственно. Внутрипакетные crossed — это порядок применения сообщений одной пачки, а не наблюдаемое рынком состояние. Отдельный подфид удалений решает: без feed3 доля crossed и внутри пачки, и на её конце взлетает до десятков процентов у всех трёх бирж.

Кроме перекрёста невозможной оказывается глубина. У Bybit и OKX восстановленный стакан держится в границах снапшота (49/49 и 399/399 уровней против 50/50 и 400/400), а у Binance распухает в среднем до 11 037 bid-уровней при снапшотных 100 — это на два порядка больше базы, максимум доходит до 14 482. Причина в том, что дельты несут новые цены за пределами снапшотной глубины, а снятий для них в дампе нет; формально получается стакан глубже, чем биржа когда-либо показывала базой. Top-of-book от этого не страдает (совпадение с эталоном 97.2 %), но глубину за 100 уровнями использовать нельзя: в пункте 4 проход по книге обрезан этой базой.

Feed health различается сильно: медианная задержка Binance $\approx$ 2.3 мс, OKX $\approx$ 23.9 мс, Bybit $\approx$ 36.8 мс; хвосты (p99) — единицы-десятки миллисекунд, отдельные выбросы доходят до сотен миллисекунд. Задержка считается с наносекундной точностью приёма, и именно поэтому медиана Binance держится на 2.3 мс: `event_time` усечён до миллисекунд, а микросекунды лежат в `event_time_nanos`; биржевой `exchange_time_nanos` при этом тождественно нулевой у всех трёх бирж, то есть наносекунды уточняют только нашу сторону. Разницу медиан (до $\approx$35 мс) вторая оценка — кросс-корреляция приростов mid — не подтверждает как расхождение часов: максимум корреляции между OKX и остальными стоит на нулевом лаге, а для пары Binance–Bybit выигрывает лаг 0.1 с (0.62 против 0.36), то есть Bybit приходит позже, а не «идёт со сдвинутыми часами».

Порядок приёма и порядок биржи расходятся редко, но систематически. Если упорядочить события по наносекундному времени приёма, биржевой штамп убывает в 0.18 % случаев у Bybit, 0.22 % у Binance и 0.08 % у OKX — это пакеты, пришедшие по разным подфидам, где более позднее по бирже событие принято раньше. Ни один из двух таймстемпов сам по себе не задаёт правильную последовательность, поэтому книга собирается в физическом порядке приёма `(event_time, file_row_number)`, а `exchange_time` служит только метрикой латентности.

Сборщик сверен с независимым эталоном `l2_executable_quotes_100ms`: совпадение обеих котировок 99.8 % у Bybit, 97.2 % у Binance и 99.9 % у OKX. Остаток объясняется дискретностью сетки эталона: смена top-of-book попадает в соседнюю 100-мс ячейку.

Симулированные исполнения при задержке 50 мс. Медианный slippage до размера 1 BTC — ровно 0.5 тика на всех биржах: это квантование на тик-сетке, а не издержка. Настоящая стоимость появляется с 5 BTC и сильно зависит от биржи: у Bybit 8.1 тика против 0.5 у Binance и OKX; на 20 BTC Bybit 22.0, Binance 26.7, OKX 37.5 тика; на 50 BTC 22.8, 21.2 и 58.8. Но на этих размерах большая часть ордеров уже частичная — глубины снимка не хватает на весь объём (на 50 BTC: Bybit 98.8 %, Binance 90.0 %, OKX 67.5 %), поэтому медианы считаются по меньшей, полностью исполняемой части и сравнение крупных размеров между биржами ограничено глубиной данных. Отрицательный slippage у крупных размеров до исключения частичных исполнений был артефактом неполного прохода. Markouts в базисных пунктах на 100 мс, 1 с и 10 с медианно отрицательны на уровне половины тика ($\approx -0.0085$ б.п. при спреде 1 тик). Среднее по обеим сторонам почти не движется с горизонтом ($-0.12\ldots-0.25$ б.п. у разных бирж) — при симметричной выборке вклады покупки и продажи сокращаются; средние по сторонам на горизонте 10 с расходятся слабо (покупка $-0.10\ldots-0.22$, продажа $-0.16\ldots-0.28$ б.п.), и этой асимметрии на сотне точек решения недостаточно, чтобы говорить о значимом adverse selection.

Набор инвариантов, считающий отклонения от чистого прогона, ловит каждую из четырёх инъекций: drop — разрывом и всплеском crossed (1 разрыв, +38 п.п.), duplicate — дубликатом сообщения и разрывом, reorder и reset — сбросом номера; на чистом куске все инварианты проходят. Все числа по издержкам — верхняя граница: собственное влияние на рынок, пассивные исполнения, busts, STP и реальные комиссии не моделируются, как и написано в футере задания.
